In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from scipy.stats import linregress
df_alpha = pd.read_csv("../data/processed/regime_alpha_dataset.csv")
prices_raw = pd.read_csv("../data/raw/prices_raw.csv")
prices_raw["timestamp"] = pd.to_datetime(prices_raw["timestamp"])
# sort by market and timestamp 
prices_raw = prices_raw.sort_values(["market_id", "timestamp"]).reset_index(drop=True)


# getting time horizont early  (25%)
def get_early_market_slice(market_df, horizon=0.25):
    market_df = market_df.sort_values("timestamp").reset_index(drop=True)
    n_obs = len(market_df)
    cutoff = max(int(np.ceil(n_obs * horizon)),2)
    early_df = market_df.iloc[:cutoff].copy()
    # Activity controls
    early_df["horizon"] = horizon
    early_df["total_n_obs"] = n_obs
    early_df["early_n_obs"] = len(early_df)
    early_df["early_obs_share"] = len(early_df) / n_obs

    # Time controls
    total_time = (market_df["timestamp"].max() - market_df["timestamp"].min())

    early_time = ( early_df["timestamp"].max() - market_df["timestamp"].min())
    early_df["total_time_seconds"] = total_time.total_seconds()
    early_df["early_time_seconds"] = early_time.total_seconds()
    early_df["early_time_share"] = np.where(early_df["total_time_seconds"] > 0,
    early_df["early_time_seconds"] / early_df["total_time_seconds"],np.nan)
    return early_df


In [2]:
horizons = [0.25, 0.50, 0.75]

early_slices = []

for horizon in horizons:
    
    temp = []

    for market_id, market_df in prices_raw.groupby("market_id"):
        early_market = get_early_market_slice(market_df,horizon=horizon)
        temp.append(early_market)

    horizon_df = pd.concat(temp,ignore_index=True)

    early_slices.append(horizon_df)

early_prices = pd.concat(early_slices,ignore_index=True)

early_prices.shape

(150996, 12)

In [3]:
early_prices.groupby("horizon")["market_id"].nunique()

horizon
0.25    43
0.50    43
0.75    43
Name: market_id, dtype: int64

In [4]:
early_prices.head(2)

,timestamp,implied_probability,market_id,question,slug,horizon,total_n_obs,early_n_obs,early_obs_share,total_time_seconds,early_time_seconds,early_time_share
0,2023-02-07 01:00:32+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.25,1826,457,0.250274,7163979.0,1825177.0,0.254771
1,2023-02-07 02:00:53+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,0.25,1826,457,0.250274,7163979.0,1825177.0,0.254771


In [5]:
from scipy.stats import linregress

def compute_basic_early_features(group, market_id, horizon):

    group = group.sort_values("timestamp").reset_index(drop=True)
    p = group["implied_probability"].astype(float)
    returns = p.diff().dropna()

    if len(p) >= 2:
        x = np.arange(len(p))
        trend = linregress(x, p).slope
    else:
        trend = np.nan

    last_p = p.iloc[-1]
    distance_to_boundary = min(last_p, 1 - last_p)

    features = {
        "market_id": market_id,
        "horizon": horizon,

        "question": group["question"].iloc[0],
        "slug": group["slug"].iloc[0],

        "total_n_obs": group["total_n_obs"].iloc[0],
        "early_n_obs": group["early_n_obs"].iloc[0],
        "early_obs_share": group["early_obs_share"].iloc[0],
        "total_time_seconds": group["total_time_seconds"].iloc[0],
        "early_time_seconds": group["early_time_seconds"].iloc[0],
        "early_time_share": group["early_time_share"].iloc[0],

        "early_first_probability": p.iloc[0],
        "early_last_probability": last_p,
        "early_mean_probability": p.mean(),
        "early_std_probability": p.std(),
        "early_probability_range": p.max() - p.min(),
        "early_realized_volatility": returns.std(),

        "early_trend": trend,

        "early_distance_to_boundary": distance_to_boundary,
        "early_mean_distance_to_boundary": np.minimum(p, 1 - p).mean(),
        "early_min_distance_to_boundary": np.minimum(p, 1 - p).min(),
        "early_near_boundary": int((last_p <= 0.05) or (last_p >= 0.95)),
    }

    return features

In [6]:
feature_rows = []

for (market_id, horizon), group in early_prices.groupby(["market_id", "horizon"]):
    
    row = compute_basic_early_features(group=group,market_id=market_id,horizon=horizon)
    feature_rows.append(row)
early_features_basic = pd.DataFrame(feature_rows)
early_features_basic.head()

,market_id,horizon,question,slug,total_n_obs,early_n_obs,early_obs_share,total_time_seconds,early_time_seconds,early_time_share,...,early_last_probability,early_mean_probability,early_std_probability,early_probability_range,early_realized_volatility,early_trend,early_distance_to_boundary,early_mean_distance_to_boundary,early_min_distance_to_boundary,early_near_boundary
0,248594,0.25,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,457,0.250274,7163979.0,1825177.0,0.254771,...,0.15,0.325470,0.225373,0.87,0.034559,-0.001320,0.15,0.299956,0.06,0
1,248594,0.50,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,913,0.500000,7163979.0,3553194.0,0.495981,...,0.10,0.214545,0.194571,0.87,0.024594,-0.000537,0.10,0.201774,0.06,0
2,248594,0.75,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,1370,0.750274,7163979.0,5392799.0,0.752766,...,0.06,0.169599,0.171271,0.88,0.020089,-0.000294,0.06,0.161088,0.05,0
3,249778,0.25,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,6,0.260870,100797.0,18011.0,0.178686,...,0.50,0.500000,0.000000,0.00,0.000000,0.000000,0.50,0.500000,0.50,0
4,249778,0.50,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,12,0.521739,100797.0,43197.0,0.428554,...,0.50,0.500000,0.000000,0.00,0.000000,0.000000,0.50,0.500000,0.50,0


In [7]:
early_features_basic.groupby("horizon")["market_id"].nunique()

horizon
0.25    43
0.50    43
0.75    43
Name: market_id, dtype: int64

In [8]:
alpha_cols = [
    "market_id",
    "outcome",
    "final_probability",
    "bayesian_fair_probability",
    "mispricing",
    "abs_mispricing",
    "abs_surprise",
    "cluster",
    "market_regime"
]

alpha_base = df_alpha[alpha_cols].copy()
early_regime_dataset = early_features_basic.merge(alpha_base,on="market_id",how="left")

early_regime_dataset.head()

,market_id,horizon,question,slug,total_n_obs,early_n_obs,early_obs_share,total_time_seconds,early_time_seconds,early_time_share,...,early_min_distance_to_boundary,early_near_boundary,outcome,final_probability,bayesian_fair_probability,mispricing,abs_mispricing,abs_surprise,cluster,market_regime
0,248594,0.25,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,457,0.250274,7163979.0,1825177.0,0.254771,...,0.06,0,0,0.02,0.038462,0.018462,0.018462,0.02,0,Information Processing
1,248594,0.50,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,913,0.500000,7163979.0,3553194.0,0.495981,...,0.06,0,0,0.02,0.038462,0.018462,0.018462,0.02,0,Information Processing
2,248594,0.75,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,1826,1370,0.750274,7163979.0,5392799.0,0.752766,...,0.05,0,0,0.02,0.038462,0.018462,0.018462,0.02,0,Information Processing
3,249778,0.25,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,6,0.260870,100797.0,18011.0,0.178686,...,0.50,0,1,0.55,0.600000,0.050000,0.050000,0.45,1,Anchored / Noisy
4,249778,0.50,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,23,12,0.521739,100797.0,43197.0,0.428554,...,0.50,0,1,0.55,0.600000,0.050000,0.050000,0.45,1,Anchored / Noisy


In [9]:
early_regime_dataset.shape

(129, 29)

In [10]:
early_regime_dataset["market_regime"].isna().sum()

np.int64(0)

In [11]:
early_regime_dataset.groupby("horizon")["market_id"].nunique()

horizon
0.25    43
0.50    43
0.75    43
Name: market_id, dtype: int64

In [12]:
# advanced features
def compute_advanced_early_features(group, market_id, horizon):
    
    group = group.sort_values("timestamp").reset_index(drop=True)
    p = group["implied_probability"].astype(float)
    returns = p.diff().dropna()

    # Max drawdown
    running_max = p.cummax()
    drawdowns = (p - running_max) / running_max.replace(0, np.nan)
    early_max_drawdown = drawdowns.min()

    # Reversals
    direction = np.sign(returns)
    direction = direction[direction != 0]
    early_reversals = (direction.diff().fillna(0) != 0).sum()

    # Entropy from return direction
    if len(direction) > 0:
        probs = direction.value_counts(normalize=True)
        early_entropy = -(probs * np.log(probs)).sum()
    else:
        early_entropy = 0

    return {
        "market_id": market_id,
        "horizon": horizon,
        "early_max_drawdown": abs(early_max_drawdown),
        "early_reversals": early_reversals,
        "early_entropy": early_entropy,
    }
# applying
advanced_rows = []

for (market_id, horizon), group in early_prices.groupby(["market_id", "horizon"]):
    
    row = compute_advanced_early_features(
        group=group,
        market_id=market_id,
        horizon=horizon
    )
    
    advanced_rows.append(row)

early_features_advanced = pd.DataFrame(advanced_rows)

early_features_advanced.head()

,market_id,horizon,early_max_drawdown,early_reversals,early_entropy
0,248594,0.25,0.935484,8,0.661563
1,248594,0.50,0.935484,24,0.693147
2,248594,0.75,0.946237,30,0.689944
3,249778,0.25,0.000000,0,0.000000
4,249778,0.50,0.000000,0,0.000000


In [13]:
early_regime_dataset = early_regime_dataset.merge(early_features_advanced,on=["market_id", "horizon"],how="left")
early_regime_dataset["early_realized_volatility"] = (early_regime_dataset["early_realized_volatility"].fillna(0))


In [15]:
pd.set_option('display.max_colwidth', None)
early_regime_dataset.groupby("horizon")[
[
    "early_last_probability",
    "early_probability_range",
    "early_realized_volatility",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"
]
].describe()

early_last_probability                                             \
                         count      mean       std     min     25%    50%   
horizon                                                                     
0.25                      43.0  0.341151  0.268194  0.0145  0.1125  0.295   
0.50                      43.0  0.365395  0.286421  0.0155  0.1160  0.315   
0.75                      43.0  0.346663  0.359833  0.0050  0.0340  0.150   

                        early_probability_range            ...  \
            75%     max                   count      mean  ...   
horizon                                                    ...   
0.25     0.5275  0.9700                    43.0  0.323314  ...   
0.50     0.5625  0.9825                    43.0  0.383058  ...   
0.75     0.6430  0.9900                    43.0  0.461581  ...   

        early_reversals        early_entropy                           \
                    75%    max         count      mean       std  min   
horizon                                                                 
0.25               41.5   89.0          43.0  0.584263  0.218158 -0.0   
0.50              110.5  221.0          43.0  0.603031  0.199637 -0.0   
0.75              222.5  656.0          43.0  0.621671  0.177905 -0.0   

                                                 
              25%       50%       75%       max  
horizon                                          
0.25     0.602034  0.678914  0.688786  0.693147  
0.50     0.628250  0.682908  0.692288  0.693147  
0.75     0.655468  0.688457  0.692599  0.693147  

[3 rows x 48 columns]

In [18]:
early_regime_dataset[numeric_cols].agg(
    ["min","max"]
).T.sort_index()

,min,max
abs_mispricing,0.011538,2.650000e-01
abs_surprise,0.000500,7.650000e-01
bayesian_fair_probability,0.038462,9.000000e-01
cluster,0.000000,1.000000e+00
early_distance_to_boundary,0.005000,5.000000e-01
early_entropy,-0.000000,6.931472e-01
early_first_probability,0.030000,9.500000e-01
early_last_probability,0.005000,9.900000e-01
early_max_drawdown,0.000000,9.901961e-01
early_mean_distance_to_boundary,0.020413,5.000000e-01


In [19]:
for col in [
    "early_probability_range",
    "early_realized_volatility",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"
]:
    
    print("\n", col)
    
    print(
        early_regime_dataset[col]
        .describe(
            percentiles=[0.01,0.05,0.95,0.99]
        )
    )


 early_probability_range
count    129.000000
mean       0.389318
std        0.229071
min        0.000000
1%         0.000000
5%         0.020000
95%        0.733800
99%        0.870000
max        0.880000
Name: early_probability_range, dtype: float64

 early_realized_volatility
count    129.000000
mean       0.026345
std        0.030068
min        0.000000
1%         0.000000
5%         0.000179
95%        0.064838
99%        0.145190
max        0.232909
Name: early_realized_volatility, dtype: float64

 early_max_drawdown
count    129.000000
mean       0.633032
std        0.287543
min        0.000000
1%         0.000000
5%         0.000000
95%        0.966667
99%        0.984359
max        0.990196
Name: early_max_drawdown, dtype: float64

 early_reversals
count    129.000000
mean      72.294574
std      125.270483
min        0.000000
1%         0.000000
5%         0.000000
95%      313.400000
99%      637.480000
max      656.000000
Name: early_reversals, dtype: float64

 early_entrop

In [20]:
early_regime_dataset[
    early_regime_dataset["early_reversals"] ==
    early_regime_dataset["early_reversals"].max()
][[
    "market_id",
    "horizon",
    "early_reversals",
    "early_n_obs"
]]

,market_id,horizon,early_reversals,early_n_obs
107,254580,0.75,656,4899


Notebook 03 — Early Regime Detection
Objetivo

El objetivo de esta notebook es responder una única pregunta:

¿Las dinámicas tempranas de un mercado contienen suficiente información para identificar su comportamiento futuro?

La idea es investigar si observando únicamente una parte inicial de la trayectoria de probabilidades podemos detectar patrones estructurales antes de que el mercado complete su ciclo de vida.

Datos Utilizados
prices_raw

Contiene la serie temporal de probabilidades para cada mercado:

timestamp
implied_probability
market_id
question
slug

Cada fila representa una observación de la probabilidad implícita en un momento determinado.

Construcción de Horizontes Tempranos

Para cada mercado:

Se ordenó la trayectoria por timestamp.
Se construyeron cortes tempranos de la vida del mercado:
25%
50%
75%

Por ejemplo:

Mercado completo

|------------------------|

25%

|------|

El objetivo es simular la información que habría estado disponible antes de la resolución del mercado.

Feature Engineering Temprano

A partir de cada trayectoria parcial se reconstruyeron métricas que describen el comportamiento del mercado en sus primeras etapas.

Dinámica de Probabilidades
early_first_probability
early_last_probability
early_mean_probability
early_std_probability
early_probability_range
early_realized_volatility
early_trend

Estas variables capturan:

Nivel promedio de creencia del mercado.
Magnitud de los movimientos.
Velocidad de convergencia.
Tendencia general de actualización de probabilidades.
Efecto Frontera (0–1)

Se incorporaron variables para capturar mercados que convergen rápidamente hacia una respuesta aparentemente resuelta.

early_distance_to_boundary
early_mean_distance_to_boundary
early_min_distance_to_boundary
early_near_boundary

Estas métricas buscan distinguir entre:

Baja volatilidad por falta de información

y

Baja volatilidad porque el mercado ya convergió hacia 0 o 1
Complejidad de la Trayectoria

También se calcularon métricas que describen la forma de la evolución temprana del mercado:

early_max_drawdown
early_reversals
early_entropy

Estas variables intentan medir:

Retrocesos importantes en la confianza del mercado.
Frecuencia de cambios de dirección.
Nivel de incertidumbre e información contenida en la trayectoria.
Controles de Actividad

Se añadieron variables para controlar diferencias en cantidad de observaciones y tiempo disponible:

early_n_obs
total_n_obs
early_obs_share

early_time_seconds
total_time_seconds
early_time_share

Esto permite distinguir mercados con mucha actividad temprana de mercados con pocas actualizaciones.

Dataset Resultante

El dataset final contiene:

43 mercados
×
3 horizontes (25%, 50%, 75%)

=
129 observaciones

Cada fila representa:

un mercado
+
un horizonte temporal
+
sus características tempranas
Validaciones Realizadas

Se verificó que:

No existieran valores infinitos.
No existieran valores faltantes relevantes.
Las variables estuvieran dentro de rangos razonables.
Las métricas de volatilidad, drawdown, reversals y entropy fueran consistentes con las trayectorias observadas.
Siguiente Paso

Con el dataset ya construido, el siguiente experimento consiste en evaluar si estas características tempranas contienen información suficiente para distinguir distintos comportamientos de mercado.

La pregunta central pasa de ser:

¿Cómo se comportó el mercado?

a

¿Qué podemos inferir sobre el mercado observando únicamente sus primeras etapas?